## Install libraries and download the dataset

In [1]:
!pip install datasets transformers rouge-score nltk SentencePiece -qqq

## Load the dataset

In [2]:
import transformers
from datasets import load_dataset, load_metric
import pandas as pd

In [3]:
dataset_file = '/home/ubuntu/Downloads/Preprocessed1_news_output.csv'

dataset = load_dataset('csv', data_files=dataset_file)

Using custom data configuration default-4bab0b219e71fafa
Reusing dataset csv (/home/ubuntu/.cache/huggingface/datasets/csv/default-4bab0b219e71fafa/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)


  0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 2225
    })
})

## Dataset train/validation/test split

In [5]:
datasets_train_test = dataset["train"].train_test_split(test_size=500)
datasets_train_validation = datasets_train_test["train"].train_test_split(test_size=500)

dataset["train"] = datasets_train_validation["train"]
dataset["validation"] = datasets_train_validation["test"]
dataset["test"] = datasets_train_test["test"]
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 1225
    })
    validation: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 500
    })
    test: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 500
    })
})

In [6]:
n_samples_train = len(dataset["train"])
n_samples_validation = len(dataset["validation"])
n_samples_test = len(dataset["test"])
n_samples_total = n_samples_train + n_samples_validation + n_samples_test

print(f"- Training set: {n_samples_train*100/n_samples_total:.2f}%")
print(f"- Validation set: {n_samples_validation*100/n_samples_total:.2f}%")
print(f"- Test set: {n_samples_test*100/n_samples_total:.2f}%")

- Training set: 55.06%
- Validation set: 22.47%
- Test set: 22.47%


In [7]:
# keep only a subsample of the datasets
dataset["train"] = dataset["train"].shuffle().select(range(1224))
dataset["validation"] = dataset["validation"].shuffle().select(range(500))
dataset["test"] = dataset["test"].shuffle().select(range(500))

dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 1224
    })
    validation: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 500
    })
    test: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert'],
        num_rows: 500
    })
})

## Data preprocessing

In [8]:
import nltk
nltk.download('punkt')
import string

from transformers import AutoTokenizer,BartTokenizer, BartModel

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
model_checkpoint = "facebook/mbart-large-50"

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

In [10]:
prefix = "summarize: "

max_input_length = 1024
max_target_length = 128

def clean_text(text):
  sentences = nltk.sent_tokenize(text.strip())
  sentences_cleaned = [s for sent in sentences for s in sent.split("\n")]
  sentences_cleaned_no_titles = [sent for sent in sentences_cleaned
                                 if len(sent) > 0 and
                                 sent[-1] in string.punctuation]
  text_cleaned = "\n".join(sentences_cleaned_no_titles)
  return text_cleaned

def preprocess_data(examples):
  texts_cleaned = [clean_text(text) for text in examples["text"]]
  inputs = [prefix + text for text in texts_cleaned]
  model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

  # Setup the tokenizer for targets
  with tokenizer.as_target_tokenizer():
    labels = tokenizer(examples["title"], max_length=max_target_length, 
                       truncation=True)

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

In [11]:
dataset_cleaned =dataset.filter(lambda example: (len(example['text']) >= 500) and (len(example['title']) >= 20))
tokenized_datasets = dataset_cleaned.map(preprocess_data, batched=True)
tokenized_datasets

  0%|          | 0/2 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/2 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1094
    })
    validation: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 436
    })
    test: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'text', 'categories', 'article_final', 'Summary_Bert', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 449
    })
})

# Fine-tune BART

In [12]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,AutoModel, EarlyStoppingCallback, IntervalStrategy

In [13]:
batch_size = 4
model_name = "BART-base-medium-title-generation"
model_dir = f"drive/MyDrive/Models/{model_name}"


In [14]:
args = Seq2SeqTrainingArguments(
    model_dir,
   f"training_with_callbacks",
   evaluation_strategy = IntervalStrategy.STEPS,
    eval_steps=100,
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps",
    save_steps=100,
    learning_rate=6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    #fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    report_to="tensorboard"
)

In [15]:
data_collator = DataCollatorForSeq2Seq(tokenizer)

In [16]:
import numpy as np

metric = load_metric("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip()))
                      for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) 
                      for label in decoded_labels]
    
    # Compute ROUGE scores
    result = metric.compute(predictions=decoded_preds, references=decoded_labels,
                            use_stemmer=True)

    # Extract ROUGE f1 scores
    result = {key: value.mid.fmeasure for key, value in result.items()}
    
    # Add mean generated length to metrics
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id)
                      for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    return {k: round(v, 4) for k, v in result.items()}

In [17]:
# Function that returns an untrained model to be trained
def model_init():
    return AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

trainer = Seq2SeqTrainer(
    model_init=model_init,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

loading configuration file https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json from cache at /home/ubuntu/.cache/huggingface/transformers/199ab6c0f28e763098fd3ea09fd68a0928bb297d0f76b9f3375e8a1d652748f9.930264180d256e6fe8e4ba6a728dd80e969493c23d4caa0a6f943614c52d34ab
Model config BartConfig {
  "_name_or_path": "facebook/bart-large-cnn",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to

In [18]:
# Start TensorBoard before training to monitor it in progress
%load_ext tensorboard
%tensorboard --logdir '/content/drive/MyDrive/Models/BART-base-medium-title-generation'/runs

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
trainer.train()

loading configuration file https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json from cache at /home/ubuntu/.cache/huggingface/transformers/199ab6c0f28e763098fd3ea09fd68a0928bb297d0f76b9f3375e8a1d652748f9.930264180d256e6fe8e4ba6a728dd80e969493c23d4caa0a6f943614c52d34ab
Model config BartConfig {
  "_name_or_path": "facebook/bart-large-cnn",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
100,1.563700,1.439373,0.560400,0.428700,0.470800,0.547600,93.091700
200,1.423200,1.383201,0.586800,0.446300,0.487000,0.572400,106.841700
300,1.318300,1.432764,0.595200,0.446900,0.484800,0.581000,121.967900
400,0.910500,1.412088,0.587900,0.433200,0.474700,0.572200,120.084900
500,0.893600,1.363990,0.595500,0.440800,0.483400,0.579600,125.376100
600,0.716200,1.547462,0.588400,0.428100,0.476400,0.572000,128.479400
700,0.513200,1.512592,0.596800,0.439200,0.482500,0.580000,127.722500
800,0.507600,1.547177,0.593600,0.432800,0.478500,0.576200,134.227100
900,0.323500,1.730265,0.595300,0.434700,0.478000,0.577400,134.213300
1000,0.282600,1.726524,0.592800,0.431100,0.477700,0.576000,134.456400


The following columns in the evaluation set  don't have a corresponding argument in `BartForConditionalGeneration.forward` and have been ignored: article_final, categories, Summary_Bert, text, title, Unnamed: 0.1, Unnamed: 0. If article_final, categories, Summary_Bert, text, title, Unnamed: 0.1, Unnamed: 0 are not expected by `BartForConditionalGeneration.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 436
  Batch size = 4
Saving model checkpoint to drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-100
Configuration saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-100/config.json
Model weights saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-100/pytorch_model.bin
tokenizer config file saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-100/tokenizer_config.json
Special tokens file saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkp

***** Running Evaluation *****
  Num examples = 436
  Batch size = 4
Saving model checkpoint to drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-800
Configuration saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-800/config.json
Model weights saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-800/pytorch_model.bin
tokenizer config file saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-800/tokenizer_config.json
Special tokens file saved in drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-800/special_tokens_map.json
Deleting older checkpoint [drive/MyDrive/Models/BART-base-medium-title-generation/checkpoint-500] due to args.save_total_limit
The following columns in the evaluation set  don't have a corresponding argument in `BartForConditionalGeneration.forward` and have been ignored: article_final, categories, Summary_Bert, text, title, Unnamed: 0.1, Unnamed: 0. If article_final, c

TrainOutput(global_step=1000, training_loss=0.8452518424987793, metrics={'train_runtime': 12875.8745, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.085, 'total_flos': 2939694323269632.0, 'train_loss': 0.8452518424987793, 'epoch': 3.65})

In [ ]:
trainer.evaluate()

The following columns in the evaluation set  don't have a corresponding argument in `BartForConditionalGeneration.forward` and have been ignored: text, Summary_Bert, categories, Unnamed: 0, title, article_final, Unnamed: 0.1. If text, Summary_Bert, categories, Unnamed: 0, title, article_final, Unnamed: 0.1 are not expected by `BartForConditionalGeneration.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 445
  Batch size = 2


In [20]:
trainer.save_model()

Saving model checkpoint to drive/MyDrive/Models/BART-base-medium-title-generation
Configuration saved in drive/MyDrive/Models/BART-base-medium-title-generation/config.json
Model weights saved in drive/MyDrive/Models/BART-base-medium-title-generation/pytorch_model.bin
tokenizer config file saved in drive/MyDrive/Models/BART-base-medium-title-generation/tokenizer_config.json
Special tokens file saved in drive/MyDrive/Models/BART-base-medium-title-generation/special_tokens_map.json


## Load the model from GDrive

In [21]:
model_name = "BART-base-medium-title-generation/checkpoint-2000"
model_dir = f"drive/MyDrive/Models/{model_name}"

tokenizer = AutoTokenizer.from_pretrained("drive/MyDrive/Models/BART-base-medium-title-generation")
model = AutoModelForSeq2SeqLM.from_pretrained("drive/MyDrive/Models/BART-base-medium-title-generation")

max_input_length = 512

Didn't find file drive/MyDrive/Models/BART-base-medium-title-generation/added_tokens.json. We won't load it.
loading file drive/MyDrive/Models/BART-base-medium-title-generation/vocab.json
loading file drive/MyDrive/Models/BART-base-medium-title-generation/merges.txt
loading file drive/MyDrive/Models/BART-base-medium-title-generation/tokenizer.json
loading file None
loading file drive/MyDrive/Models/BART-base-medium-title-generation/special_tokens_map.json
loading file drive/MyDrive/Models/BART-base-medium-title-generation/tokenizer_config.json
loading configuration file drive/MyDrive/Models/BART-base-medium-title-generation/config.json
Model config BartConfig {
  "_name_or_path": "drive/MyDrive/Models/BART-base-medium-title-generation",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  

In [22]:
text = """
We define access to a Streamlit app in a browser tab as a session.
For each browser tab that connects to the Streamlit server, a new session is created.
Streamlit reruns your script from top to bottom every time you interact with your app.
Each reruns takes place in a blank slate: no variables are shared between runs.
Session State is a way to share variables between reruns, for each user session.
In addition to the ability to store and persist state, Streamlit also exposes the
ability to manipulate state using Callbacks. In this guide, we will illustrate the
usage of Session State and Callbacks as we build a stateful Counter app.
For details on the Session State and Callbacks API, please refer to our Session
State API Reference Guide. Also, check out this Session State basics tutorial
video by Streamlit Developer Advocate Dr. Marisa Smith to get started:
"""

inputs = ["summarize: " + text]

inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, return_tensors="pt")
output = model.generate(**inputs, num_beams=8, do_sample=True, min_length=10, max_length=64)
decoded_output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
predicted_title = nltk.sent_tokenize(decoded_output.strip())[0]

print("predicted_summary : " , predicted_title)
# Session State and Callbacks in Streamlit

predicted_summary :  Streamlit reruns your script from top to bottom every time you interact with your app.


In [23]:
df=pd.DataFrame
df=pd.read_csv('/content/drive/MyDrive/Preprocessed1_news_output.csv',skiprows = [i for i in range(1, 1726) ])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Preprocessed1_news_output.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
for i, entry in enumerate(df['text']):

  inputs = ["summarize: " + entry]

  inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, return_tensors="pt")
  output = model.generate(**inputs, num_beams=8, do_sample=True, min_length=10, max_length=64)
  decoded_output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
  predicted_title = nltk.sent_tokenize(decoded_output.strip())[0]
  df.loc[i,'Summary_BART'] = predicted_title
  print(i)
  print(predicted_title)

In [ ]:
!pip install rouge
from rouge import Rouge
import matplotlib.pyplot as plt


In [ ]:
def calculate_rouge_PRF(score):
  rouge_1 = { 'r': 0, 'p': 0, 'f': 0 }
  rouge_2 = { 'r': 0, 'p': 0, 'f': 0 }
  rouge_l = { 'r': 0, 'p': 0, 'f': 0 }
  total_len = len(scores)
  results = []

  for score in scores:
      rouge_1['r'] += score['rouge-1']['r']
      rouge_1['p'] += score['rouge-1']['p']
      rouge_1['f'] += score['rouge-1']['f']
      rouge_2['r'] += score['rouge-2']['r']
      rouge_2['p'] += score['rouge-2']['p']
      rouge_2['f'] += score['rouge-2']['f']
      rouge_l['r'] += score['rouge-l']['r']
      rouge_l['p'] += score['rouge-l']['p']
      rouge_l['f'] += score['rouge-l']['f']

  for total_score in [rouge_1, rouge_2, rouge_l]:
      for key in total_score:
          total_score[key] = total_score[key] / total_len           #divide by 2225
      results.append(total_score)
          
  df_avg_result = pd.DataFrame({ 'rouge-1': results[0], 'rouge-2': results[1], 'rouge-l': results[2] })
  return df_avg_result

In [ ]:
rouge = Rouge()
scores = rouge.get_scores(df["title"], df["Summary_BART"])
df_result = pd.DataFrame(scores)
df_result.head()
Rouge_BERT=calculate_rouge_PRF(scores)
Rouge_BERT


In [ ]:
Rouge_BERT.plot.bar()
plt.legend(loc='lower right')
plt.title("BART")